# Validación de las tablas del lakehouse

Cuaderno pensado como **última actividad de la canalización** `pl_ingesta_ventas`, después del flujo de datos Gen2. Comprueba que las tablas que el flujo cargó en el lakehouse son coherentes y coinciden con los valores esperados ([`../datos-esperados/valores-control.csv`](../datos-esperados/valores-control.csv)). Si algo falla, **lanza una excepción**: la actividad queda en rojo, la canalización falla y se ve en el centro de supervisión (*Monitor*).

**En Fabric**, el cuaderno lee las tablas del lakehouse predeterminado que tenga asociado. **En local**, para comprobar el propio cuaderno, carga como tablas los resultados esperados que genera la réplica en pandas.

In [1]:
try:
    import notebookutils  # solo existe dentro de Microsoft Fabric
    EN_FABRIC = True
except ImportError:
    EN_FABRIC = False

if not EN_FABRIC:
    from pyspark.sql import SparkSession
    spark = (SparkSession.builder.master('local[2]').appName('validacion-fabric')
             .config('spark.sql.shuffle.partitions', '4').config('spark.ui.showConsoleProgress', 'false').getOrCreate())
    for t in ['fact_ventas', 'dim_producto', 'dim_cliente', 'dim_fecha', 'ventas_mensuales']:
        (spark.read.option('header', True).option('inferSchema', True)
         .csv(f'../datos-esperados/{t}.csv').createOrReplaceTempView(t))

print(f'Entorno: {"Microsoft Fabric" if EN_FABRIC else "local"} · Spark {spark.version}')

Entorno: local · Spark 4.0.1


In [2]:
# Valores esperados (de la réplica en pandas). En Fabric, pega aquí los mismos números
# o léelos de un archivo en la sección Files del lakehouse.
ESPERADO = {
    'fact_ventas': (542, 708690.07),
    'dim_producto': (295, None),
    'dim_cliente': (847, None),
    'dim_fecha': (365, None),
    'ventas_mensuales': (122, 708690.07),
}

In [3]:
from pyspark.sql import functions as F

fact = spark.table('fact_ventas')
n = {t: spark.table(t).count() for t in ESPERADO}
suma_fact = round(fact.agg(F.sum('LineItemTotal')).first()[0], 2)
suma_mensual = round(spark.table('ventas_mensuales').agg(F.sum('Ventas')).first()[0], 2)


def huerfanos(columna, dimension, clave):
    return fact.join(spark.table(dimension), fact[columna] == spark.table(dimension)[clave], 'left_anti').count()


comprobaciones = [(f'{t}: {esperadas} filas', n[t] == esperadas, f'{n[t]} filas')
                  for t, (esperadas, _) in ESPERADO.items() if esperadas is not None]
comprobaciones += [
    ('Suma de ventas de fact_ventas', abs(suma_fact - ESPERADO['fact_ventas'][1]) < 0.01, f'{suma_fact:,.2f}'),
    ('ventas_mensuales cuadra con fact_ventas', abs(suma_mensual - suma_fact) < 0.01, f'{suma_mensual:,.2f}'),
    ('Sin duplicados (SalesOrderID, LineItem)', fact.groupBy('SalesOrderID', 'LineItem').count().filter('count > 1').count() == 0, ''),
    ('Sin nulos en claves de fact_ventas', fact.filter('SalesOrderID IS NULL OR ProductID IS NULL OR CustomerID IS NULL OR DateKey IS NULL').count() == 0, ''),
    ('Productos sin huérfanos', huerfanos('ProductID', 'dim_producto', 'ProductID') == 0, ''),
    ('Clientes sin huérfanos', huerfanos('CustomerID', 'dim_cliente', 'CustomerID') == 0, ''),
    ('Fechas sin huérfanas', huerfanos('DateKey', 'dim_fecha', 'DateKey') == 0, ''),
]

for nombre, ok, detalle in comprobaciones:
    print(f"{'[OK]   ' if ok else '[FALLO]'} {nombre:42} {detalle}")

fallidas = [nombre for nombre, ok, _ in comprobaciones if not ok]
if fallidas:
    raise AssertionError(f'Validación fallida: {fallidas}')
print('\nTodas las comprobaciones superadas')

[OK]    fact_ventas: 542 filas                     542 filas
[OK]    dim_producto: 295 filas                    295 filas
[OK]    dim_cliente: 847 filas                     847 filas
[OK]    dim_fecha: 365 filas                       365 filas
[OK]    ventas_mensuales: 122 filas                122 filas
[OK]    Suma de ventas de fact_ventas              708,690.07
[OK]    ventas_mensuales cuadra con fact_ventas    708,690.07
[OK]    Sin duplicados (SalesOrderID, LineItem)    
[OK]    Sin nulos en claves de fact_ventas         
[OK]    Productos sin huérfanos                    
[OK]    Clientes sin huérfanos                     
[OK]    Fechas sin huérfanas                       

Todas las comprobaciones superadas
